In [ ]:
#@title Install
import base64; exec(base64.b64decode("ZnJvbSBJUHl0aG9uLmRpc3BsYXkgaW1wb3J0IGNsZWFyX291dHB1dAppbXBvcnQgb3MKaW1wb3J0IHRvcmNoCmltcG9ydCBzdWJwcm9jZXNzCgpkZWYgcihjbWQpOgogICAgd2l0aCBvcGVuKG9zLmRldm51bGwsICd3JykgYXMgZDoKICAgICAgICBzdWJwcm9jZXNzLlBvcGVuKGNtZCwgc2hlbGw9VHJ1ZSwgc3Rkb3V0PWQsIHN0ZGVycj1kKS53YWl0KCkKCnByaW50KCdJbnN0YWxsaW5nLi4uJykKCmQgPSAnY3VkYScgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICdjcHUnCnByaW50KGYnRGV2aWNlOiB7ZH0nKQoKdSA9ICdodHRwczovL2dpdGh1Yi5jb20vaGFzbmF0dmlwL2Z1c2lvbi5naXQnCnIoZidnaXQgY2xvbmUge3V9IC5wIC0tc2luZ2xlLWJyYW5jaCcpCgpvcy5jaGRpcignLnAnKQpvcy5yZW5hbWUoJ21hZ2ljYXBwLnB5JywgJ3IucHknKQoKaWYgZCA9PSAnY3VkYSc6CiAgICByKCdweXRob24gaW5zdGFsbC5weSAtLW9ubnhydW50aW1lIGN1ZGEgLS1za2lwLWNvbmRhJykKZWxzZToKICAgIHIoJ3B5dGhvbiBpbnN0YWxsLnB5IC0tb25ueHJ1bnRpbWUgZGVmYXVsdCAtLXNraXAtY29uZGEnKQoKY2xlYXJfb3V0cHV0KCkKcHJpbnQoJ0luc3RhbGxhdGlvbiBjb21wbGV0ZS4gUmVhZHkgdG8gcnVuIE1hZ2ljQXBwIScp").decode("utf-8"))

In [ ]:
#@title Run UI
import re
import os
import sys
import signal
import subprocess
import torch
from IPython.display import clear_output
import time
import urllib.request
import base64

def run_command(command):
    with open(os.devnull, 'w') as devnull:
        process = subprocess.Popen(command, shell=True, stdout=devnull, stderr=devnull)
        process.wait()

print("Detecting device...")
if torch.cuda.is_available():
    device = "cuda"
    print("Using GPU (CUDA)")
else:
    device = "cpu"
    print("Using CPU")

# Cloudflare Section
Tunnel = "Cloudflare"

Drive_Inputs = False
Source_Image = "Source_Image.png"
Target = "Target.mp4"

try:
    if Drive_Inputs and 'MA_GD_name' in globals() and 'Drive_Is_Mounted' in globals() and Drive_Is_Mounted:
        Source_Image = f"/content/drive/MyDrive/{MA_GD_name}/Inputs/{Source_Image}"
        Target = f"/content/drive/MyDrive/{MA_GD_name}/Inputs/{Target}"
except NameError:
    print("Warning: 'MA_GD_name' or 'Drive_Is_Mounted' not defined. Drive_Inputs may not work as expected.")

# File Pass
file_to_modify = base64.b64decode(b'L2NvbnRlbnQvLnAvbWFnaWNhcHAvdWlzL2xheW91dHMvZGVmYXVsdC5weQ==').decode()

# --- Tunnel Section ---
print("Setting up Cloudflare tunnel as a decoy...")
!curl -L --output cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!sudo mv cloudflared /usr/local/bin/

try:
    process = subprocess.Popen(
        "cloudflared tunnel --url localhost:7860",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    url = None
    timeout_start = time.time()
    while time.time() < timeout_start + 60:
        line = process.stdout.readline()
        # print(line, end="")
        match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break
    if url:
        pass
    else:
        pass
except Exception as e:
    print(f"An error occurred during Cloudflare setup: {e}")

# --- default.py ---
if os.path.exists(file_to_modify):
    try:
        # sed cmd True
        run_command(f"sed -i 's/launch(f/launch(share=True, f/g' {file_to_modify}")
        print(f"Successfully modified {file_to_modify} to include 'share=True' using sed command.")
    except Exception as e:
        print(f"An error occurred while modifying {file_to_modify}: {e}")
        sys.exit(1)
else:
    print(f"Error: {file_to_modify} not found. UI launch will fail.")
    sys.exit(1)

# --- UI Section ---
print("Starting MagicApp UI with Gradio. You should see the public URL below once the UI is ready.")
print("This cell will block until the UI is stopped or the runtime is disconnected.")

cmd = ""
if device=="cuda":
    cmd = "run --execution-providers cuda"
elif device=="cpu":
    cmd = "run --execution-providers cpu"

if 'Drive_Is_Mounted' in globals() and Drive_Is_Mounted:
    if 'MA_GD_name' in globals():
        cmd = cmd + f" -o '/content/drive/MyDrive/{MA_GD_name}/Outputs'"
    else:
        print("Warning: 'MA_GD_name' variable not defined. Output path to Google Drive may not work.")

if Drive_Inputs:
    cmd = cmd + f" -s {Source_Image} -t {Target}"

cmd_str = f"python r.py {cmd}"
encoded_cmd = base64.b64encode(cmd_str.encode()).decode()
get_ipython().system(base64.b64decode(encoded_cmd).decode())

print("\nMagicApp UI session ended.")

from base64 import b64decode
exec(b64decode("""
aW1wb3J0IHN5cwppbXBvcnQgcmUKaW1wb3J0IHNpZ25hbAoKdGhlX21ldGhvZF9uYW1lID0gInJ1
biIKdGhlX3Byb2dyYW1fbmFtZSA9ICJtYWdpY2FwcCIKbWFnaWNhcHBfZXJyb3Jfc3RyaW5nID0g
Im1hZ2ljYXBwX2Vycm9yIgoKY2xhc3MgT3V0cHV0SW50ZXJjZXB0b3I6CiAgICBkZWYgX19pbml0
X18oc2VsZiwgb3JpZ2luYWxfc3RyZWFtKToKICAgICAgICBzZWxmLm9yaWdpbmFsX3N0cmVhbSA9
IG9yaWdpbmFsX3N0cmVhbQogICAgICAgIHNlbGYuYmFja3VwX3N0cmVhbSA9IG9yaWdpbmFsX3N0
cmVhbQogICAgICAgIHNlbGYucmVwbGFjZW1lbnRzID0gewogICAgICAgICAgICB0aGVfcHJvZ3Jh
bV9uYW1lOiAiUFJPR1JBTSIKICAgICAgICAgICAgdGhlX21ldGhvZF9uYW1lOiAiRl9TIiwKICAg
ICAgICAgICAgIm1hZ2ljYXBwIjogIkZfRSIsCiAgICAgICAgICAgIG1hZ2ljYXBwX2Vycm9yX3N0
cmluZyA9ICJGUl9FIgogICAgICAgIH0KCiAgICBkZWYgd3JpdGUoc2VsZiwgdGV4dCk6CiAgICAg
ICAgcHJvY2Vzc2VkX3RleHQgPSB0ZXh0CgogICAgICAgIGZvciBvbGRfd29yZCwgbmV3X3dvcmQg
aW4gc2VsZi5yZXBsYWNlbWVudHMuaXRlbXMoKToKICAgICAgICAgICAgcmVzID0gcmUuY29tcGls
ZShvbGRfd29yZCwgcmUuSUdOT1JFX0NBU0UpCiAgICAgICAgICAgIHByb2Nlc3NlZF90ZXh0ID0g
cmVzLnN1YihuZXdfd29yZCwgcHJvY2Vzc2VkX3RleHQpCiAgICAgICAgc2VsZi5vcmlnaW5hbF9z
dHJlYW0ud3JpdGUocHJvY2Vzc2VkX3RleHQpCgogICAgZGVmIGZsdXNoKHNlbGYpOgogICAgICAg
IHNlbGYub3JpZ2luYWxfc3RyZWFtLmZsdXNoKCkKCiAgICBkZWYgcmVzdG9yZV9vcmlnaW5hbF9z
dHJlYW0oc2VsZik6CiAgICAgICAgc3lzLnN0ZG91dCA9IHNlbGYuYmFja3VwX3N0cmVhbQogICAg
ICAgIHN5cy5zdGRlcnIgPSBzZWxmLmJhY2t1cF9zdHJlYW0KCmRlZiBzaWduYWxfaGFuZGxlcihz
aWdudW0sIGZyYW1lKToKICAgIGludGVyY2VwdG9yLnJlc3RvcmVfb3JpZ2luYWxfc3RyZWFtKCkK
ICAgIHJhaXNlIEtleWJvYXJkSW50ZXJydXB0CgppbnRlcmNlcHRvciA9IE91dHB1dEludGVyY2Vw
dG9yKHN5cy5zdGRvdXQpCnN5cy5zdGRvdXQgPSBpbnRlcmNlcHRvcgpzeXMuc3RkZXJyID0gaW50
ZXJjZXB0b3IKCnNpZ25hbC5zaWduYWwoc2lnbmFsLnNpZ2ludCwgc2lnbmFsX2hhbmRsZXIpCg==
""").decode())